# MVO Notebook 验证

本文件使用独立的新 Python 内核运行测试，避免依赖当前 Notebook 中残留的变量。默认先执行核心代码 Notebook，再执行 `mvo.tests.ipynb`。

运行 `./run_tests.sh --notebooks` 会设置环境变量 `MVO_FULL_VALIDATION=1`，依次完整执行核心代码、API 演示、真实数据实验及测试。完整执行需要数分钟，首次运行需要安装依赖及下载数据。结果记录在 `docs/validation.json`，执行过的 Notebook 会保留输出。


In [1]:
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform

import nbformat
from nbclient import NotebookClient

ROOT = Path.cwd()
assert (ROOT / "mvo_utils.ipynb").is_file(), "请从项目目录运行本 Notebook。"
FULL_VALIDATION = os.environ.get("MVO_FULL_VALIDATION", "0") == "1"
notebook_names = ["mvo_utils.ipynb"]
if FULL_VALIDATION:
    notebook_names += ["mvo.API.ipynb", "mvo.example.ipynb"]
notebook_names.append("mvo.tests.ipynb")
print("本次执行：", notebook_names)


本次执行： ['mvo_utils.ipynb', 'mvo.API.ipynb', 'mvo.example.ipynb', 'mvo.tests.ipynb']


## 1. 记录本次源码和验证范围

哈希只覆盖各单元格的类型及源码，不包含输出和执行编号。因此重新运行产生的时间或图表不会改变源码标识。执行前先写入 `running` 状态，失败时不会把旧的成功记录误当作本次结果。


In [2]:
def notebook_source_hash(path):
    notebook = nbformat.read(path, as_version=4)
    source = [{"cell_type": cell.cell_type, "source": cell.source}
              for cell in notebook.cells]
    encoded = json.dumps(source, ensure_ascii=False, sort_keys=True).encode()
    return hashlib.sha256(encoded).hexdigest()


def save_validation(report):
    (ROOT / "docs").mkdir(exist_ok=True)
    (ROOT / "docs/validation.json").write_text(json.dumps(report, indent=2) + "\n")


report = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "running",
    "scope": "full" if FULL_VALIDATION else "tests_only",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "planned_notebooks": notebook_names,
    "executed_notebooks": [],
    "unit_and_integration_tests": "pending",
    "source_hash_method": "SHA-256 of JSON cell_type/source, ensure_ascii=False, sort_keys=True",
    "notebook_sources": [{"path": path.name, "sha256": notebook_source_hash(path)}
                         for path in sorted(ROOT.glob("*.ipynb"))],
}
save_validation(report)


## 2. 在新内核中运行

每份 Notebook 都先清除旧输出，再从第一格执行。失败时保存已产生的输出和失败状态，并向上抛出错误；只有全部成功才继续。


In [3]:
for name in notebook_names:
    print(f"Executing {name} in a fresh kernel", flush=True)
    path = ROOT / name
    notebook = nbformat.read(path, as_version=4)
    for cell in notebook.cells:
        if cell.cell_type == "code":
            cell.outputs = []
            cell.execution_count = None
    try:
        NotebookClient(notebook, timeout=1800, resources={
            "metadata": {"path": str(ROOT)}
        }).execute()
    except Exception as error:
        report.update(status="failed", failed_notebook=name, error=str(error))
        save_validation(report)
        raise
    finally:
        nbformat.write(notebook, path)
    report["executed_notebooks"].append(name)
    if name == "mvo.tests.ipynb":
        report["unit_and_integration_tests"] = "passed"
        report["tests_run"] = 15
    save_validation(report)
    print(f"Passed: {name}", flush=True)


Executing mvo_utils.ipynb in a fresh kernel


Passed: mvo_utils.ipynb


Executing mvo.API.ipynb in a fresh kernel


Passed: mvo.API.ipynb


Executing mvo.example.ipynb in a fresh kernel


Passed: mvo.example.ipynb


Executing mvo.tests.ipynb in a fresh kernel


Passed: mvo.tests.ipynb


## 3. 保存环境与最终状态

这份记录明确区分仅测试与完整实验验证。`mvo_utils.ipynb` 由每份演示或测试 Notebook 的 `%run` 单元格执行。


In [4]:
packages = ["numpy", "pandas", "scipy", "torch", "cvxpy", "cvxpylayers",
            "scikit-learn", "diffcp", "clarabel", "scs", "matplotlib",
            "nbformat", "nbclient"]
report["packages"] = {name: importlib.metadata.version(name) for name in packages}
report["validated_at_utc"] = datetime.now(timezone.utc).isoformat()
report["status"] = "passed"
save_validation(report)
print("Project validation passed.")
print("验证范围：", report["scope"])
print("已通过：", report["executed_notebooks"])


Project validation passed.
验证范围： full
已通过： ['mvo_utils.ipynb', 'mvo.API.ipynb', 'mvo.example.ipynb', 'mvo.tests.ipynb']
